# Thí nghiệm mất cân bằng lớp (tuần 4, thành viên A)

Thiết kế và cách phân tích: `experiments/imbalance/README.md`. Baseline là `yolov8s_1280` của B (không train lại).

**Mỗi lần chạy notebook chỉ train MỘT mô hình.** Đổi `RUN` ở cell đầu:
- `"imb_rfs"`: oversampling lớp hiếm (Repeat Factor Sampling, thr = 0.3)
- `"imb_clspw"`: trọng số lớp trong loss (cls_pw = 0.5)

Trước khi chạy: Settings → Accelerator **GPU T4** và **Internet: On**; Add Input → dataset `za-traffic-2020`; Add-ons → Secrets → `WANDB_API_KEY`.

In [ ]:
RUN = "imb_rfs"      # đổi thành "imb_clspw" cho lần chạy thứ hai
SMOKE = False        # True = chạy thử 1 epoch trên 5% dữ liệu để kiểm tra pipeline (vài phút)

assert RUN in ("imb_rfs", "imb_clspw")
NAME = RUN + ("_smoke" if SMOKE else "")
print("Sẽ train:", NAME)

In [ ]:
# Ghim đúng phiên bản B đã dùng cho baseline yolov8s_1280
!pip install -q "ultralytics==8.4.204" pycocotools wandb
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert ultralytics.__version__ == "8.4.204", "Sai phiên bản ultralytics, hãy Restart kernel rồi chạy lại cell này"

In [ ]:
from pathlib import Path
%cd /kaggle/working
if not Path("deeplearning-project-sign-detection").exists():
    !git clone https://github.com/tmcng/deeplearning-project-sign-detection.git
%cd /kaggle/working/deeplearning-project-sign-detection
!git pull -q
!git log --oneline -1

In [ ]:
# Tìm thư mục ảnh gốc (giống notebook của B)
IMAGES_DIR = None
for p in Path("/kaggle/input").rglob("traffic_train/images"):
    if p.is_dir():
        IMAGES_DIR = p.resolve()
        break
assert IMAGES_DIR is not None, "Không thấy traffic_train/images. Add Input -> dataset za-traffic-2020"
print(IMAGES_DIR, "|", len(list(IMAGES_DIR.glob("*.png"))), "ảnh")

In [ ]:
import os
WANDB_FLAG = "--no-wandb"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    import wandb
    wandb.login(key=os.environ["WANDB_API_KEY"])
    WANDB_FLAG = ""
    print("Đã đăng nhập W&B")
except Exception as e:
    print("Không có WANDB_API_KEY, chạy không ghi W&B:", e)

In [ ]:
# Dữ liệu YOLO (symlink tới ảnh gốc, không tốn dung lượng) + file oversampling
!python yolo/coco_to_yolo.py --splits data/splits --images "{IMAGES_DIR}" --out data/yolo
!python experiments/imbalance/oversample_yolo.py --yolo data/yolo --thr 0.3
# Kiểm tra: train phải có 2933 ảnh, 5693 box; oversampling 2933 ảnh -> 3619 dòng

In [ ]:
# Train: tham số giống hệt lệnh của B cho yolov8s_1280, chỉ khác dữ liệu hoặc cls_pw
data = "data/yolo/data_rfs.yaml" if RUN == "imb_rfs" else "data/yolo/data.yaml"
extra = "--cls-pw 0.5" if RUN == "imb_clspw" else ""
epochs = "--epochs 1 --fraction 0.05" if SMOKE else "--epochs 50"
!python yolo/train_yolov8.py --data {data} --model yolov8s.pt --imgsz 1280 --batch 8 {epochs} {extra} --name {NAME} --exist-ok {WANDB_FLAG}
# Với imb_clspw: log phải có dòng "Class weights: [...]"

In [ ]:
# Dự đoán trên val (suy luận thường ở 1280, giống B) và chấm điểm
pts = sorted(Path("runs").rglob(f"{NAME}/weights/best.pt"))
assert pts, "Không tìm thấy best.pt"
BEST = pts[-1]
print("Dùng", BEST)
!python yolo/predict_yolo.py --weights "{BEST}" --gt data/splits/val.json --images data/yolo/images/val     --imgsz 1280 --out preds/{NAME}_val.json --eval --name {NAME}

In [ ]:
# Gom kết quả vào một file zip để tải về (tab Output bên phải, hoặc mục Output của version đã lưu)
import shutil, zipfile
out = Path(f"/kaggle/working/ket_qua_{NAME}.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [Path(f"preds/{NAME}_val.json"), Path(f"reports/eval/{NAME}.json"), Path("reports/eval/results.md"), BEST]:
        z.write(f, f.as_posix())
    for f in BEST.parent.parent.glob("*"):
        if f.is_file():                      # results.csv, args.yaml, biểu đồ, confusion matrix của Ultralytics
            z.write(f, f"run/{f.name}")
print("Đã tạo", out, round(out.stat().st_size / 1e6, 1), "MB")
shutil.rmtree("data/yolo", ignore_errors=True)  # bỏ thư mục dữ liệu khỏi Output cho gọn